# Dissociated scRNA-seq with a paired ST section

Dissociated cells have **no tissue coordinates**. DeltaTrace infers them from a paired spatial
transcriptomics (ST) section: map every cell onto the section's spots, then take the Earth
Mover's (1-Wasserstein) distance between two cells' spot distributions as their physical distance.

**map** (any probabilistic mapper; Tangram here) -> `pp.wasserstein` -> `pp.graphs` -> `tl.trajectory`

This is a template: point the two paths below at your own data and set the cluster names.

In [ ]:
import scanpy as sc

import deltatrace as dt

## Load

- `adata_sc`: the dissociated cells, **raw counts**, with a cluster / cell-type label in `obs`.
- `adata_st`: the paired ST section, raw counts, spot coordinates in `obsm["spatial"]`.

In [ ]:
adata_sc = sc.read_h5ad("path/to/sc.h5ad")
adata_st = sc.read_h5ad("path/to/st.h5ad")

cluster_key = "cell_type"                  # column of adata_sc.obs holding the labels
sources = ["progenitor"]                   # where the trajectory starts
sinks = ["fate_A", "fate_B"]               # the terminal fates (see source-only below if unknown)
pxl_to_um = 1.0                            # ST coordinate units -> microns

## 1. Map cells to spots

DeltaTrace takes any probabilistic cell-to-spot mapping, it only needs an `AnnData` of shape `(n_cells, n_spots)` whose rows are distributions over `adata_st`'s spots (matched by spot name when both carry names). Already have one? Load it as `ad_map` and skip this cell.

Here we use [Tangram](https://doi.org/10.1038/s41592-021-01264-7), is not a DeltaTrace dependency (to install run: `pip install tangram-sc`).

In [ ]:
import tangram as tg

sc_map, st_map = adata_sc.copy(), adata_st.copy()
sc.pp.normalize_total(sc_map)
sc.pp.normalize_total(st_map)
tg.pp_adatas(sc_map, st_map, genes=None)   # None: every shared gene; marker genes map better
ad_map = tg.map_cells_to_space(sc_map, st_map, device="cpu")   # "cuda:0" on a GPU

## 2. Cell-cell spatial distance

The Earth Mover's distance between every pair of cells' spot distributions, over the spots'
Euclidean distances. Every pair is solved, so this is the slow step on large datasets
(`n_jobs=-1`, the default, uses every core).

In [ ]:
adata_dist = dt.pp.wasserstein(ad_map, adata_st, pxl_to_um=pxl_to_um)

## 3. Expression graph + integration

Standard scanpy preprocessing; DeltaTrace reads the kNN graph `sc.pp.neighbors` leaves in the
object. `pp.graphs` then builds the spatially integrated graphs next to it. UMAP is only for plotting.

In [ ]:
sc.pp.normalize_total(adata_sc)
sc.pp.log1p(adata_sc)
sc.pp.highly_variable_genes(adata_sc, n_top_genes=2000)
sc.pp.pca(adata_sc)
sc.pp.neighbors(adata_sc)
sc.tl.umap(adata_sc)

ad_intg = dt.pp.graphs(adata_sc, adata_dist)

## 4. Trajectory and lineages

One call samples source and sink cells, solves the transport for pseudotime and flow, builds
the velocity field (on the UMAP here: dissociated cells have no tissue coordinates) and assigns
every cell a fate.

In [ ]:
dt.tl.trajectory(ad_intg, sources, sinks, cluster_key=cluster_key, graph="intr")

dt.pl.pseudotime(ad_intg)

Each cell's fate is in `obs["lineage"]`, with `obs["lineage_confidence"]` and a
`fate_<lineage>` probability per fate. Cells the flow never carries to a declared sink are
`"unassigned"` (drawn gray).

In [ ]:
dt.pl.lineages(ad_intg)
ad_intg.obs["lineage"].value_counts()

The same flow, summarized between clusters:

In [ ]:
dt.tl.cluster_trajectory(ad_intg, cluster_key=cluster_key)
dt.pl.cluster_trajectory(ad_intg, cluster_key=cluster_key)

## 5. Source-only: when the terminal fates are unknown

With `sink_clusters=None`, every non-source cell absorbs the flow, and `trajectory` prints how
terminal each cluster ends up (`Term = 1 - outflow/inflow`) instead of assigning lineages: you
pick the fates. `key_added` keeps this run next to the one above in the same object.

Source-only is for *finding* the fates: once you know them, pass them as `sinks` as in step 4,
which gives the better trajectory.

In [ ]:
run = "source_only"
res = dt.tl.trajectory(ad_intg, sources, None, cluster_key=cluster_key, graph="intr", key_added=run)
res["terminality"]

In [ ]:
# lineages on this run's flow, cut at the number of fates the ranking shows
dt.tl.lineages(ad_intg, sources, cluster_key=cluster_key, n_lineages=2, key_added=run)
dt.pl.lineages(ad_intg, key_added=run)